# 04 - Memory Management & Troubleshooting

This notebook covers Spark's memory internals and common failure patterns:

1. **Memory model** - Unified Memory Manager, execution vs storage memory, dynamic borrowing
2. **OOM patterns** - driver OOM (collect/toPandas), executor OOM (skew, large broadcast)
3. **Tungsten & codegen** - binary format, whole-stage codegen, why Python UDFs hurt codegen and what alternatives exist

Dataset: NYC Yellow Taxi Trip Records (2024).

In [1]:
from pathlib import Path
from pyspark.sql import functions as F
from pyspark.sql.functions import udf
import time

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
TAXI_DATA_DIR = DATA_DIR / "taxi"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)

PROJECT_ROOT: C:\code\spark-tuning-handbook
DATA_DIR: C:\code\spark-tuning-handbook\data


In [2]:
taxi = spark.read.parquet(str(TAXI_DATA_DIR))
print("rows:", taxi.count())
print("partitions:", taxi.rdd.getNumPartitions())

rows: 41169720
partitions: 6


Spark UI: __http://localhost:4040__

We use Spark UI in this lab after each action to validate:
* execution vs storage memory usage per executor
* spill (memory/disk) in task metrics
* GC time pressure
* whether driver or executor failed (OOM pattern)

In [7]:
# Keep plans deterministic for learning cells
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.shuffle.partitions", "8")

print("spark.sql.adaptive.enabled:", spark.conf.get("spark.sql.adaptive.enabled"))
print("spark.sql.shuffle.partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

spark.sql.adaptive.enabled: false
spark.sql.shuffle.partitions: 8


## Helpers

Reusable functions for tracking jobs/stages in Spark UI and inspecting physical plans.

In [11]:
def run_and_report(action_label, action_fn):
    """Executes an action under a unique job group ID (visible in Spark UI), then collects and prints the job and stage IDs it produced."""
    sc = spark.sparkContext
    tracker = sc.statusTracker()
    group_id = f"notebook_demo_{int(time.time() * 1000)}"
    sc.setJobGroup(group_id, action_label)
    try:
        result = action_fn()
    finally:
        job_ids = list(tracker.getJobIdsForGroup(group_id))
        stage_ids = set()
        for job_id in job_ids:
            job_info = tracker.getJobInfo(job_id)
            if job_info is not None:
                stage_ids.update(list(job_info.stageIds))
        print(
            f"{action_label} -> jobs={job_ids}, stages={sorted(stage_ids)}, stage_count={len(stage_ids)}"
        )
        sc.setJobGroup("", "")
    return result


def show_exchange_nodes(df):
    plan = df._jdf.queryExecution().executedPlan().toString()
    lines = [line.strip() for line in plan.splitlines() if "Exchange" in line]
    if not lines:
        print("No Exchange nodes in executed physical plan.")
    else:
        print("Exchange-related nodes:")
        for line in lines:
            print(line)

## Section 1 - Spark Memory Model and UMM

### Unified Memory Manager (UMM)

Each executor's JVM heap is divided into:
- **Reserved memory** (300 MB) - fixed, internal Spark overhead
- **User memory** (1 - `spark.memory.fraction`) - user data structures, UDF variables, RDD metadata
- **Unified memory** (`spark.memory.fraction`, default 0.6) - shared pool split into execution and storage memory

**Execution memory** - buffers for sort, shuffle, join hash tables, aggregation. When operators exceed available space, Spark spills to disk from here.

**Storage memory** - cache/persist blocks and broadcast variables.

Key concepts:
- **JVM heap** - memory allocated by Java at executor startup, sized by `spark.executor.memory`. Everything Spark does in an executor lives here
- **GC (Garbage Collector)** - Java's automatic memory cleanup. While GC runs, the executor pauses (**stop-the-world**). More cached objects on heap => more GC pressure => slower tasks
- **memoryOverhead** - memory outside JVM heap for Python workers, network buffers, container overhead. Not subject to GC.
- **Off-heap** - additional memory outside JVM, disabled by default (`spark.memory.offHeap.enabled`). Tungsten can store data here, bypassing GC entirely

### Dynamic borrowing and eviction

- execution can borrow from idle storage
- storage can use idle execution space
- but execution can **evict** storage blocks when it needs memory back - storage cannot evict execution
- this means **execution always has priority** => if a sort or shuffle needs memory, cached data gets dropped first
- evicted cached blocks are either recomputed or read from disk (if `StorageLevel` includes disk)


### When things go wrong

- **High GC Time in Spark UI** - too much cached data on heap, or many short-lived objects from wide operations (e.g. `groupBy` on millions of keys creates temporary hash maps that GC must clean up). **Fix**: increase heap, reduce cache, or use serialized storage level.
- **Spill (memory -> disk)** - execution memory full, but Spark handles it gracefully by writing to disk. Job finishes, just slower. **Fix**: reduce partition skew or increase partitions.
- **OOM** - heap completely exhausted, spill cannot save it (e.g. single object too large to fit, or memory fragmented beyond recovery). Executor or driver crashes. **Fix**: distinguish driver vs executor OOM (next section).

### Demo - execution evicts cached data

Cache a DataFrame into storage memory, then run a heavy aggregation + sort that pressures execution memory. Compare cache read time before and after - if eviction happened, the second read is slower (recompute from parquet).

Check Spark UI -> Storage tab to see cached partitions drop.

In [20]:
spark.catalog.clearCache()

cached_mid = (
    taxi.select(
        "PULocationID", "DOLocationID", "fare_amount", "tip_amount",
        "total_amount", "trip_distance", "passenger_count",
        "tpep_pickup_datetime", "tpep_dropoff_datetime"
    )
    .repartition(24, "PULocationID")
    .cache()
)

run_and_report("cache materialize", lambda: cached_mid.count())

cache materialize -> jobs=[11], stages=[26, 27, 28], stage_count=3


41169720

In [18]:
run_and_report("cache read BEFORE pressure", lambda: cached_full.count())

cache read BEFORE pressure -> jobs=[9], stages=[20, 21, 22], stage_count=3


41169720

In [15]:
# Heavy groupBy + sort to pressure execution memory
pressure = (
    cached_trips
    .groupBy("PULocationID", "DOLocationID")
    .agg(F.count("*").alias("trip_count"), F.avg("total_amount").alias("avg_total"))
    .orderBy(F.desc("trip_count"))
)

run_and_report("execution pressure", lambda: pressure.limit(10).collect())

execution pressure -> jobs=[6], stages=[12, 13], stage_count=2


[Row(PULocationID=237, DOLocationID=236, trip_count=279894, avg_total=15.438311682273437),
 Row(PULocationID=236, DOLocationID=237, trip_count=241425, avg_total=15.895152117629896),
 Row(PULocationID=237, DOLocationID=237, trip_count=196947, avg_total=13.601488623841655),
 Row(PULocationID=236, DOLocationID=236, trip_count=191380, avg_total=13.010062859233079),
 Row(PULocationID=161, DOLocationID=237, trip_count=131903, avg_total=17.06480390893284),
 Row(PULocationID=237, DOLocationID=161, trip_count=117112, avg_total=17.25461566705428),
 Row(PULocationID=161, DOLocationID=236, trip_count=109747, avg_total=21.879319343581347),
 Row(PULocationID=237, DOLocationID=162, trip_count=107499, avg_total=16.21815486655788),
 Row(PULocationID=239, DOLocationID=142, trip_count=105687, avg_total=14.279177003795667),
 Row(PULocationID=142, DOLocationID=239, trip_count=105221, avg_total=14.658590015301778)]

In [16]:
run_and_report("cache read AFTER pressure", lambda: cached_trips.count())

cache read AFTER pressure -> jobs=[7], stages=[14, 15, 16], stage_count=3


41169720

## 2) Diagnosing OOM: driver vs executor

Driver OOM pattern:
- full `collect()`/`toPandas()` on large frames

Executor OOM pattern:
- skewed partitions, giant hash tables, oversized broadcast build side

Use plans + stage/task metrics + key distribution before changing configs.


In [ ]:
risk = taxi.select("PULocationID","DOLocationID","fare_amount","tip_amount","total_amount")
print("estimated collect payload MB", round(est_collect_mb(risk),1))
rows, t_take = run_and_report("safe bounded take", lambda: risk.take(20))
print("rows returned", len(rows), "seconds", round(t_take,2))
if RUN_DANGEROUS_FAILURES:
    try:
        run_and_report("dangerous full collect", lambda: risk.collect())
    except Exception as e:
        print("expected failure path", type(e).__name__)
else:
    print("dangerous collect skipped")


In [ ]:
spark.conf.set("spark.sql.shuffle.partitions", "8")
skew = taxi.select("PULocationID","DOLocationID","total_amount").withColumn("k", F.when((F.col("PULocationID")%25)==0, F.col("PULocationID")).otherwise(F.lit(0)))
skew.groupBy("k").count().orderBy(F.desc("count")).show(10, False)
skew_agg = skew.groupBy("k").agg(F.avg("total_amount").alias("avg_total"))
show_nodes(skew_agg)
_, t_skew = run_and_report("skewed agg", lambda: skew_agg.count())

salt_n = 16
salted = skew.withColumn("salt", F.when(F.col("k")==0, (F.rand(seed=42)*salt_n).cast("int")).otherwise(F.lit(0)))
partial = salted.groupBy("k","salt").agg(F.avg("total_amount").alias("a"), F.count("*").alias("n"))
fixed = partial.groupBy("k").agg((F.sum(F.col("a")*F.col("n"))/F.sum("n")).alias("avg_total"))
_, t_salt = run_and_report("salted agg", lambda: fixed.count())
print("skew", round(t_skew,2), "salted", round(t_salt,2), "factor", round(t_skew/max(t_salt,1e-9),2))


In [ ]:
fact = taxi.select("PULocationID","DOLocationID","total_amount")
small = taxi.select("PULocationID","RatecodeID","Airport_fee").dropDuplicates(["PULocationID"])
bhj = fact.join(F.broadcast(small), on="PULocationID", how="inner")
show_nodes(bhj)
_, t_bhj = run_and_report("BHJ small side", lambda: bhj.count())

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
smj = fact.join(small, on="PULocationID", how="inner")
show_nodes(smj)
_, t_smj = run_and_report("SMJ fallback", lambda: smj.count())
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10*1024*1024))
print("BHJ", round(t_bhj,2), "SMJ", round(t_smj,2))


In [ ]:
under = taxi.select("PULocationID","total_amount").coalesce(2).groupBy("PULocationID").agg(F.avg("total_amount").alias("a"))
well = taxi.select("PULocationID","total_amount").repartition(24,"PULocationID").groupBy("PULocationID").agg(F.avg("total_amount").alias("a"))
_, t_under = run_and_report("under-partitioned", lambda: under.count())
_, t_well = run_and_report("key repartitioned", lambda: well.count())
print("under", round(t_under,2), "well", round(t_well,2))


## 3) Tungsten, off-heap, codegen, and UDF cost

Tungsten uses binary in-memory format and code-generated operator pipelines.

Why Python UDF hurts:
- adds JVM-Python serialization boundary
- appears as `BatchEvalPython` in plans
- breaks whole-stage codegen fusion across boundary


In [ ]:
for k in ["spark.memory.offHeap.enabled","spark.memory.offHeap.size","spark.sql.codegen.wholeStage","spark.sql.execution.arrow.pyspark.enabled"]:
    print(k, "=", spark.conf.get(k, "<unset>"))

@udf("double")
def tip_ratio_py(tip, fare):
    if fare is None or fare == 0 or tip is None:
        return 0.0
    return float(tip)/float(fare)

base2 = taxi.select("PULocationID","fare_amount","tip_amount").filter(F.col("fare_amount")>0)
builtin = base2.withColumn("r", F.col("tip_amount")/F.col("fare_amount"))
pyudf = base2.withColumn("r", tip_ratio_py(F.col("tip_amount"), F.col("fare_amount")))
print("builtin nodes")
show_nodes(builtin)
print("python udf nodes")
show_nodes(pyudf)
_, t_b = run_and_report("builtin agg", lambda: builtin.groupBy("PULocationID").agg(F.avg("r")).count())
_, t_p = run_and_report("python udf agg", lambda: pyudf.groupBy("PULocationID").agg(F.avg("r")).count())
print("builtin", round(t_b,2), "python_udf", round(t_p,2), "factor", round(t_p/max(t_b,1e-9),2))


In [ ]:
try:
    import pandas as pd
    from pyspark.sql.functions import pandas_udf
    @pandas_udf("double")
    def tip_ratio_pd(t, f):
        ff = f.replace(0, pd.NA)
        return (t.astype("float64")/ff.astype("float64")).fillna(0.0)
    pdu = base2.withColumn("r", tip_ratio_pd(F.col("tip_amount"), F.col("fare_amount")))
    show_nodes(pdu)
    _, t_pd = run_and_report("pandas udf agg", lambda: pdu.groupBy("PULocationID").agg(F.avg("r")).count())
    print("pandas_udf", round(t_pd,2))
except Exception as e:
    print("pandas udf skipped", e)


## Controlled failure examples
- Dangerous driver collect: disabled by default
- Forced oversized broadcast: keep as explicit opt-in

## Troubleshooting checklist
- Driver OOM -> remove unbounded collect, aggregate in cluster
- Executor OOM -> inspect skew and partition sizing first
- Broadcast instability -> lower threshold or disable broadcast
- `BatchEvalPython` in hot path -> prefer built-ins, then Pandas UDF

## Key takeaways
- Memory failures are usually partition and plan shape issues.
- UMM behavior must be tested under both storage and execution pressure.
- Tuning is empirical: explain plan + runtime metrics + controlled experiments.
